In [8]:
import os

os.environ['api_key'] = 'key'

In [2]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (964 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current use

In [3]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [4]:
!ollama -v
!ollama pull llama3.2
!ollama list

ollama version is 0.20.2

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [9]:
import openai
import requests
import sqlite3
from datetime import datetime

class WeatherAgent:
    def __init__(self):
        self.api_key = os.getenv('api_key')

    def get_weather(self, location, date):
        # Use current weather since we're dealing with same-day recommendations
        url = f"http://api.weatherapi.com/v1/current.json"
        params = {
            "key": self.api_key,
            "q": location,
            "aqi": "no"
        }
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            raise Exception(f"Weather API error: {str(e)}")

class EventAgent:
    def get_events(self, date, event_type=None):
        conn = sqlite3.connect('events.db')
        c = conn.cursor()

        try:
            if event_type:
                c.execute('SELECT * FROM events WHERE date = ? AND type = ?', (date, event_type))
            else:
                c.execute('SELECT * FROM events WHERE date = ?', (date,))

            events = c.fetchall()
            return events
        except sqlite3.Error as e:
            raise Exception(f"Database error: {str(e)}")
        finally:
            conn.close()

class RecommendationAgent:
    def __init__(self):
        # Using OpenAI client to interact with the local Ollama server running Llama3.2
        self.client = openai.OpenAI(
            base_url='http://localhost:11434/v1/',
            api_key='ollama', # Required but ignored by Ollama
        )

    def generate_recommendation(self, weather_data, events):
        # Create context for GPT
        try:
            # Handle both current and forecast data
            if 'current' in weather_data:
                weather_condition = weather_data['current']['condition']['text']
                temperature = weather_data['current']['temp_c']
                context = f"Weather: {weather_condition}, Temperature: {temperature}°C\n\n"
            else:
                context = "Weather data unavailable\n\n"

            context += "Available events:\n"
            for event in events:
                context += f"- {event[1]} ({event[2]}): {event[3]} at {event[4]}\n"

            response = self.client.chat.completions.create(
                model="llama3.2",
                messages=[
                    {"role": "system", "content": """You are a helpful event recommender. Consider the weather conditions
                    and suggest suitable events. For outdoor events, consider the temperature and weather conditions.
                    Be specific about why you recommend certain events over others. Keep your response concise but informative.
                    If weather data is unavailable, focus on providing a balanced recommendation of both indoor and outdoor events."""},
                    {"role": "user", "content": context}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            raise Exception(f"Recommendation error: {str(e)}")

class CoordinatorAgent:
    def __init__(self):
        self.weather_agent = WeatherAgent()
        self.event_agent = EventAgent()
        self.recommendation_agent = RecommendationAgent()

    def get_recommendations(self, location, date):
        try:
            # Get weather data
            print(f"\nFetching weather data for {location} on {date}...")
            weather_data = self.weather_agent.get_weather(location, date)

            # Get events
            print("Fetching events...")
            events = self.event_agent.get_events(date)

            if not events:
                return "No events found for this date."

            # Generate recommendations
            print("Generating recommendations...")
            recommendations = self.recommendation_agent.generate_recommendation(
                weather_data, events
            )

            return recommendations

        except Exception as e:
            return f"Error: {str(e)}"

In [6]:
# Base setup - Create events database:

import sqlite3

def setup_database():
    conn = sqlite3.connect('events.db')
    c = conn.cursor()

    c.execute('''
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY,
            name TEXT,
            type TEXT,  -- 'indoor' or 'outdoor'
            description TEXT,
            location TEXT,
            date TEXT
        )
    ''')

    # Sample events
    events = [
        ('Summer Concert', 'outdoor', 'Live music in the park', 'Central Park', '2025-07-15'),
        ('Art Exhibition', 'indoor', 'Modern art showcase', 'City Gallery', '2025-07-15'),
        ('Food Festival', 'outdoor', 'International cuisine', 'Waterfront', '2025-07-16'),
        ('Theater Show', 'indoor', 'Classical drama', 'Grand Theater', '2025-07-16')
    ]

    c.executemany('INSERT OR IGNORE INTO events (name, type, description, location, date) VALUES (?,?,?,?,?)', events)
    conn.commit()
    conn.close()

if __name__ == "__main__":
    setup_database()

In [10]:
test_cases = [
        ("2025-07-15", "Singapore"),
        ("2025-07-16", "Singapore"),
        ("2025-07-17", "Singapore"),  # No events on this date
    ]

coordinator = CoordinatorAgent()

for test_date, test_location in test_cases:
    print(f"\n{'='*50}")
    print(f"Getting recommendations for {test_location} on {test_date}:")
    print(f"{'='*50}")
    print(coordinator.get_recommendations(test_location, test_date))


Getting recommendations for Singapore on 2025-07-15:

Fetching weather data for Singapore on 2025-07-15...
Fetching events...
Generating recommendations...
Given the weather conditions, I would recommend either the Art Exhibition (indoor) or rescheduling the Summer Concert (outdoor).

The extreme heat and heavy rain with thunder make outdoor events hazardous, not to mention uncomfortable. Central Park's lush greenery will be under stress due to sustained rainfall, and attendees may seek shelter.

On the other hand, the Art Exhibition at City Gallery presents a cozy, climate-controlled environment that can offer an attractive respite from the inclement weather. The artwork will likely remain unaffected by the rain, providing a consistent atmosphere for art appreciation.

If indoor events are not available or preferred, consider alternatives that do not involve outdoor exposure, such as comedy shows or indoor theater performances.

The Summer Concert could be rescheduled to avoid this w

In [24]:
# Improve recommendations

import datetime
import sqlite3

def setup_database():
    conn = sqlite3.connect('events1.db')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY,
            name TEXT,
            type TEXT,
            description TEXT,
            location TEXT,
            date TEXT,
            time TEXT,
            is_outdoor INTEGER
        )
    ''')

    # Clear existing data and insert new sample data
    c.execute('DELETE FROM events')

    today = datetime.date.today()
    day1 = today.strftime('%Y-%m-%d')
    day2 = (today + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
    day3 = (today + datetime.timedelta(days=2)).strftime('%Y-%m-%d')

    events_data = [
        ("Outdoor Music Festival", "music", "Live bands all day", "Gardens by the Bay", day1, "18:00", 1),
        ("Art Exhibition", "art", "Modern art display", "National Gallery", day1, "10:00", 0),
        ("Cooking Class", "food", "Learn to cook local dishes", "Culinary Institute", day1, "14:00", 0),
        ("Night Safari Tour", "wildlife", "Explore wildlife at night", "Singapore Zoo", day1, "19:30", 1),

        ("Beach Volleyball Tournament", "sports", "Fun in the sun", "Sentosa Beach", day2, "09:00", 1),
        ("Indoor Rock Climbing", "sports", "Challenge your limits", "Climb Central", day2, "11:00", 0),
        ("Science Centre Workshop", "education", "Interactive science exhibits", "Science Centre", day2, "13:00", 0),
        ("Sunset Cruise", "leisure", "Enjoy the evening skyline", "Marina Bay", day2, "17:00", 1),

        ("Morning Yoga", "wellness", "Outdoor yoga session", "Botanic Gardens", day3, "07:00", 1),
        ("Movie Marathon", "entertainment", "Classic film screenings", "The Projector", day3, "12:00", 0),
        ("Food Fair", "food", "International cuisine stalls", "Civic District", day3, "17:00", 1)
    ]
    c.executemany('INSERT INTO events (name, type, description, location, date, time, is_outdoor) VALUES (?, ?, ?, ?, ?, ?, ?)', events_data)
    conn.commit()
    conn.close()
    print("Database 'events.db' setup complete with sample data.")

# Call the setup function to ensure the database is ready
setup_database()

2026-04-07 2026-04-08
Database 'events.db' setup complete with sample data.


In [22]:
#Implementation update to handle enhanced recommendations

import openai
import requests
import sqlite3
from datetime import datetime

class EventAgent1:
    def get_events(self, date, event_type=None):
        conn = sqlite3.connect('events1.db')
        c = conn.cursor()

        try:
            if event_type:
                c.execute('SELECT * FROM events WHERE date = ? AND type = ?', (date, event_type))
            else:
                c.execute('SELECT * FROM events WHERE date = ?', (date,))

            events = c.fetchall()
            return events
        except sqlite3.Error as e:
            raise Exception(f"Database error: {str(e)}")
        finally:
            conn.close()

class RecommendationAgent1:
    def __init__(self):
        # Using OpenAI client to interact with the local Ollama server running Llama3.2
        self.client = openai.OpenAI(
            base_url='http://localhost:11434/v1/',
            api_key='ollama', # Required but ignored by Ollama
        )

    def generate_recommendation(self, weather_data, grouped_events):
        # Create context for GPT
        try:
            # Handle both current and forecast data
            if 'current' in weather_data:
                weather_condition = weather_data['current']['condition']['text']
                temperature = weather_data['current']['temp_c']
                context = f"Weather: {weather_condition}, Temperature: {temperature}°C\n\n"
            else:
                context = "Weather data unavailable\n\n"

            context += "Available events grouped by time of day:\n"
            for time_of_day, events_list in grouped_events.items():
                if events_list:
                    context += f"\n{time_of_day} Events:\n"
                    for event in events_list:
                        context += f"- {event[1]} ({'Outdoor' if event[7] else 'Indoor'} - {event[2]}): {event[3]} at {event[5]}\n"

            print(f'Context - {context}')
            response = self.client.chat.completions.create(
                model="llama3.2",
                messages=[
                    {"role": "system", "content": """You are a helpful event recommender. Consider the weather conditions,
                    the time of day for events, and suggest suitable events. For outdoor events, consider the temperature and weather conditions.
                    Provide transport suggestions based on the weather conditions (e.g., if it's raining, suggest public transport or sheltered routes).
                    Be specific about why you recommend certain events over others. Keep your response concise but informative.
                    If weather data is unavailable, focus on providing a balanced recommendation of both indoor and outdoor events.
                    Also, suggest alternative dates for outdoor events if the weather is unfavorable, by checking other available events on nearby dates, if present in the context."""},
                    {"role": "user", "content": context}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            raise Exception(f"Recommendation error: {str(e)}")

class CoordinatorAgent1:
    def __init__(self):
        self.weather_agent = WeatherAgent()
        self.event_agent = EventAgent1()
        self.recommendation_agent = RecommendationAgent1()

    def _group_events_by_time_of_day(self, events):
        grouped = {
            "Morning": [],
            "Afternoon": [],
            "Evening": []
        }
        for event in events:
            event_time_str = event[6] # 'HH:MM' format
            if event_time_str:
                hour = int(event_time_str.split(':')[0])
                if hour < 12:
                    grouped["Morning"].append(event)
                elif 12 <= hour < 17:
                    grouped["Afternoon"].append(event)
                else:
                    grouped["Evening"].append(event)
        return grouped

    def get_recommendations(self, location, date):
        try:
            # Get weather data
            print(f"\nFetching weather data for {location} on {date}...")
            weather_data = self.weather_agent.get_weather(location, date)

            # Get events
            print("Fetching events...")
            events = self.event_agent.get_events(date)

            if not events:
                return "No events found for this date."

            # Group events by time of day
            grouped_events = self._group_events_by_time_of_day(events)

            # Generate recommendations
            print("Generating recommendations...")
            recommendations = self.recommendation_agent.generate_recommendation(
                weather_data, grouped_events
            )

            return recommendations

        except Exception as e:
            return f"Error: {str(e)}"

In [25]:
#Tests for enhanced recommendations
from datetime import date, timedelta

today = date.today()
day1 = today.strftime('%Y-%m-%d')
day2 = (today + timedelta(days=1)).strftime('%Y-%m-%d')
day3 = (today + timedelta(days=2)).strftime('%Y-%m-%d')
day4 = (today + timedelta(days=3)).strftime('%Y-%m-%d')

print(day1, day2)

test_cases = [
        (day1, "Singapore"),
        (day2, "Singapore"),
        (day3, "Singapore"),
        (day4, "Singapore") # No events on this date
    ]

coordinator = CoordinatorAgent1()

for test_date, test_location in test_cases:
    print(f"\n{'='*50}")
    print(f"Getting recommendations for {test_location} on {test_date}:")
    print(f"{'='*50}")
    print(coordinator.get_recommendations(test_location, test_date))

2026-04-07 2026-04-08

Getting recommendations for Singapore on 2026-04-07:

Fetching weather data for Singapore on 2026-04-07...
Fetching events...
Generating recommendations...
Context - Weather: Moderate or heavy rain with thunder, Temperature: 24.2°C

Available events grouped by time of day:

Morning Events:
- Art Exhibition (Indoor - art): Modern art display at 2026-04-07

Afternoon Events:
- Cooking Class (Indoor - food): Learn to cook local dishes at 2026-04-07

Evening Events:
- Outdoor Music Festival (Outdoor - music): Live bands all day at 2026-04-07
- Night Safari Tour (Outdoor - wildlife): Explore wildlife at night at 2026-04-07

Considering the weather conditions, I would recommend attending either the Art Exhibition or the Cooking Class. Both events are indoors and can be enjoyed safely.

However, considering your morning preferences, I suggest the Art Exhibition on April 7th. It's a wonderful opportunity to explore modern art in an indoor setting, protected from the rain